<a href="https://colab.research.google.com/github/alkadevirajbanshi-ai/Intelligence-System/blob/main/Travel_Agent_Minimal_Assessment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Enhancing Agent Capabilities with Generative AI
## Practical: Tool Calling with a Free Local LLM (and Optional OpenAI)

**Environment:** Google Colab (T4 GPU recommended; CPU possible but slower)  
**Default model:** `qwen2.5:3b` served locally with Ollama;

### Learning outcomes
1. Explain the difference between a tool *schema*, a model *tool-call request*, and actual Python *tool execution*.
2. Create a JSON-schema function tool for a simulated flight-booking operation.
3. Implement a multi-turn assistant that asks for missing booking information.
4. Execute tool calls and return tool results to the model with correct message history.
5. Compare local inference with the optional paid OpenAI configuration.

> **Important:** This tutorial simulates booking. No live flight data is searched, no reservation is created, and no payment is made. Colab free-tier resources are limited and not guaranteed. Ollama's model download requires internet access.

## Part A — Set up the local model

We use the **OpenAI Python SDK** with Ollama's **OpenAI-compatible local endpoint**.

This keeps the request and response structures nearly identical to the openai.chat.completions.create(...)` example.

1. In Colab, choose **Runtime → Change runtime type → T4 GPU** if available.
2. Run the installation and server-start cells below.
3. No OpenAI API key is needed for the default route.

In [1]:
%pip -q install -U openai ipywidgets requests

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


In [2]:
# Install zstd and pciutils (for lspci GPU detection) and then install Ollama
!sudo apt-get update && sudo apt-get install -y zstd pciutils
!curl -fsSL https://ollama.com/install.sh | sh

Get:1 https://cli.github.com/packages stable InRelease [3,917 B]
Get:2 https://cloud.r-project.org/bin/linux/ubuntu noble-cran40/ InRelease [3,631 B]
Get:3 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2404/x86_64  InRelease [1,578 B]
Hit:4 http://archive.ubuntu.com/ubuntu noble InRelease
Get:5 http://archive.ubuntu.com/ubuntu noble-updates InRelease [126 kB]
Get:6 https://cloud.r-project.org/bin/linux/ubuntu noble-cran40/ Packages [78.0 kB]
Get:7 http://security.ubuntu.com/ubuntu noble-security InRelease [126 kB]
Get:8 https://r2u.stat.illinois.edu/ubuntu noble InRelease [9,161 B]
Get:9 https://cli.github.com/packages stable/main amd64 Packages [357 B]
Get:10 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2404/x86_64  Packages [2,044 kB]
Get:11 http://archive.ubuntu.com/ubuntu noble-backports InRelease [126 kB]
Get:12 https://r2u.stat.illinois.edu/ubuntu noble/main all Packages [10.5 MB]
Get:13 http://archive.ubuntu.com/ubuntu noble-updates/universe am

In [3]:
# set up local LLM
import subprocess, time, requests, os

# Launch Ollama in the background; re-running this cell is safe.
if not requests.get("http://127.0.0.1:11434/api/tags", timeout=2).ok if False else False:
    pass
try:
    requests.get("http://127.0.0.1:11434/api/tags", timeout=2).raise_for_status()
    print("Ollama is already running")
except Exception:
    ollama_process = subprocess.Popen(
        ["ollama", "serve"], stdout=open("/tmp/ollama.log", "a"),
        stderr=subprocess.STDOUT
    )
    for _ in range(30):
        try:
            requests.get("http://127.0.0.1:11434/api/tags", timeout=2).raise_for_status()
            print("Ollama server ready")
            break
        except Exception:
            time.sleep(1)
    else:
        raise RuntimeError("Ollama did not start; inspect /tmp/ollama.log")

Ollama server ready


In [4]:
# Download the open-weight instruction model (once per new Colab runtime).
!ollama pull qwen2.5:3b

### Choose a model provider

**Default:** local Ollama; no paid API calls. The **optional OpenAI block** is commented out so students with credits can activate it. ChatGPT subscriptions do not include OpenAI API credits.

In [5]:
from openai import OpenAI

# DEFAULT: free, locally served Qwen model via Ollama
MODEL = "qwen2.5:3b"
client = OpenAI(
    base_url="http://127.0.0.1:11434/v1",
    api_key="ollama"  # Dummy string required by SDK; not a paid key
)

# OPTIONAL: paid OpenAI API (uncomment ALL lines below and comment
# out the local MODEL/client lines above).
# import getpass
# MODEL = "gpt-4o-mini"
# client = OpenAI(api_key=getpass.getpass("Enter OpenAI API key: "))

print("Selected model:", MODEL)

Selected model: qwen2.5:3b


In [6]:
# Smoke test: basic inference before introducing tools
reply = client.chat.completions.create(
    model=MODEL,
    messages=[{"role": "user", "content": "Reply with exactly: Travel agent ready."}],
    temperature=0,
)
print(reply.choices[0].message.content)

Travel agent ready.


## Part B — Define the tool functions

This function is **not called automatically by the LLM**. The model generates a request to call it; the agent controller validates the arguments and invokes it. This makes the application logic seprate from the tool calling.

In [7]:
from datetime import date
from typing import Dict

BOOKING_LOG = []

def book_flight(passenger_name: str, from_city: str,
                to_city: str, travel_date: str) -> dict:
    """Simulate a booking. No real ticket is purchased."""
    if not all(isinstance(x, str) and x.strip()
               for x in (passenger_name, from_city, to_city, travel_date)):
        return {"status": "error", "message": "All four fields are required."}
    try:
        parsed = date.fromisoformat(travel_date)
    except ValueError:
        return {"status": "error", "message": "Use date format YYYY-MM-DD."}
    if parsed < date.today():
        return {"status": "error", "message": "Travel date must not be in the past."}
    record = {
        "status": "simulated_only",
        "passenger_name": passenger_name.strip(),
        "from_city": from_city.strip(),
        "to_city": to_city.strip(),
        "travel_date": travel_date,
        "message": "SIMULATION ONLY: no flight has actually been booked."
    }
    BOOKING_LOG.append(record)
    return record

# Direct function test (no model/API call)
print(book_flight("Test Student", "London", "Paris", "2030-06-15"))
BOOKING_LOG.clear()  # Keep the example clean

{'status': 'simulated_only', 'passenger_name': 'Test Student', 'from_city': 'London', 'to_city': 'Paris', 'travel_date': '2030-06-15', 'message': 'SIMULATION ONLY: no flight has actually been booked.'}


## Part C — Describe the tool using JSON Schema

The JSON schema tells the model which function exists and which parameters it must supply. `required` declares mandatory fields; it does **not** guarantee the model will wait for them. We enforce that in our system instructions and validate in Python.

In [8]:
tools = [{
    "type": "function",
    "function": {
        "name": "book_flight",
        "description": (
            "Simulate a flight booking only after the passenger has supplied "
            "their full name, departure city, destination city and future travel date. "
            "This does NOT make a real reservation."
        ),
        "parameters": {
            "type": "object",
            "properties": {
                "passenger_name": {"type": "string", "description": "Passenger full legal name"},
                "from_city": {"type": "string", "description": "Departure city"},
                "to_city": {"type": "string", "description": "Destination city"},
                "travel_date": {"type": "string", "description": "Travel date in YYYY-MM-DD format"},
            },
            "required": ["passenger_name", "from_city", "to_city", "travel_date"],
            "additionalProperties": False,
        },
    },
}]
print("Tool name:", tools[0]["function"]["name"])

Tool name: book_flight


## Part D — Build the tool-calling agent

**Agent loop:** user message → model → either follow-up question **or** tool-call request → Python execution → tool result → model confirmation. We preserve the assistant tool-call message and each `tool_call_id` to keep conversation history valid.

A smaller local model may occasionally request tools prematurely or produce malformed arguments. The controller below validates all tool inputs and returns errors as tool results. We also instruct the model to ask questions instead of guessing.

In [9]:
import json

SYSTEM_PROMPT = """You are a friendly travel assistant demonstrating function calling.
All bookings are simulated; no actual flight or hotel reservation is made.
Support flight booking, hotel search, hotel booking, activities and cancellation.
Use the current customer details from the conversation; never assume a name or city.
For hotel search ask for location, maximum nightly price and Wi-Fi preference.
For hotel booking ask for passenger name, hotel name, city, check-in and check-out dates.
For cancellation ask only for the booking ID.
Before calling book_flight, obtain all four details from the user:
(1) passenger full legal name, (2) departure city, (3) destination city,
(4) future travel date in YYYY-MM-DD format.
Ask a concise follow-up question for missing details. Do not invent details.
Do not claim a real booking was made. When the tool returns an error, explain
what needs to be corrected. After a successful tool result, clearly say
'SIMULATED booking' and summarize only the returned details.
"""

def new_conversation():
    return [{"role": "system", "content": SYSTEM_PROMPT}]

messages = new_conversation()

TOOL_REGISTRY = {"book_flight": book_flight}

def travel_agent(user_message: str, messages: list, max_rounds: int = 4) -> str:
    """Multi-turn tool-calling loop compatible with local Ollama or OpenAI."""
    messages.append({"role": "user", "content": user_message})
    for step in range(max_rounds):
        response = client.chat.completions.create(
            model=MODEL,
            messages=messages,
            tools=tools,
            tool_choice="auto",
            temperature=0,
        )
        assistant = response.choices[0].message
        # Append SDK message as a serializable dictionary.
        messages.append(assistant.model_dump(exclude_none=True))

        if not assistant.tool_calls:
            return assistant.content or "Please provide more details."

        for call in assistant.tool_calls:
            name = call.function.name
            print(f"[TRACE] Model requested tool: {name}")
            try:
                args = json.loads(call.function.arguments)
                if not isinstance(args, dict):
                    raise ValueError("Tool arguments must be a JSON object")
                if name not in TOOL_REGISTRY:
                    raise ValueError(f"Unknown tool: {name}")
                # FIX: Each tool has its OWN required arguments from its JSON schema.
                # Previously flight fields were enforced for hotels and cancellations.
                schema = next(t["function"]["parameters"] for t in tools
                              if t["function"]["name"] == name)
                required = schema.get("required", [])
                missing = [key for key in required
                           if key not in args or args[key] is None
                           or (isinstance(args[key], str) and not args[key].strip())]
                if missing:
                    result = {"status": "error", "message": f"Missing: {', '.join(missing)}. Ask the user."}
                else:
                    result = TOOL_REGISTRY[name](**args)
            except Exception as exc:
                result = {"status": "error", "message": str(exc)}
            print("[TRACE] Tool result:", result)
            messages.append({
                "role": "tool",
                "tool_call_id": call.id,
                "name": name,
                "content": json.dumps(result),
            })
    return "The agent reached its tool-call limit. Please try again with full details."


## Part E — Test the follow-up questions and tool call

Run the cells one by one. Watch the trace: no Python tool should execute until all details are provided. **LLM outputs vary**; if it asks a different follow-up question, that is acceptable.

In [10]:
messages = new_conversation()
print("AGENT:", travel_agent("I want to book a flight.", messages))

AGENT: Could you please provide me with your full name, the departure city, the destination city, and the travel date in YYYY-MM-DD format?


In [11]:
print("AGENT:", travel_agent("My name is Alex Morgan. I am flying from London to Paris.", messages))

[TRACE] Model requested tool: book_flight
[TRACE] Tool result: {'status': 'error', 'message': 'Travel date must not be in the past.'}
AGENT: The travel date you provided is in the past. Could you please provide a future travel date?


In [12]:
print("AGENT:", travel_agent("Travel date is 2030-06-15.", messages))
print("SIMULATED BOOKINGS:", len(BOOKING_LOG))

[TRACE] Model requested tool: book_flight
[TRACE] Tool result: {'status': 'simulated_only', 'passenger_name': 'Alex Morgan', 'from_city': 'London', 'to_city': 'Paris', 'travel_date': '2030-06-15', 'message': 'SIMULATION ONLY: no flight has actually been booked.'}
AGENT: SIMULATION ONLY: no flight has actually been booked. Your details have been simulated for booking purposes. You will be flying from London to Paris on 2030-06-15.
SIMULATED BOOKINGS: 1


### Inspect conversation history

Find `assistant` messages containing `tool_calls`, followed by matching `tool` messages. This is the essential protocol for tool use.

In [13]:
for idx, msg in enumerate(messages):
    print(idx, msg.get("role"),
          "tool_calls=" + str(bool(msg.get("tool_calls"))),
          "tool_call_id=" + str(msg.get("tool_call_id", "")))

0 system tool_calls=False tool_call_id=
1 user tool_calls=False tool_call_id=
2 assistant tool_calls=False tool_call_id=
3 user tool_calls=False tool_call_id=
4 assistant tool_calls=True tool_call_id=
5 tool tool_calls=False tool_call_id=call_ubocoaat
6 assistant tool_calls=False tool_call_id=
7 user tool_calls=False tool_call_id=
8 assistant tool_calls=True tool_call_id=
9 tool tool_calls=False tool_call_id=call_9m7316av
10 assistant tool_calls=False tool_call_id=


## Part F — Chat interface

This interface uses `ipywidgets`. It keeps a single conversation state until **Reset chat** is clicked. HTML output is escaped to prevent user-entered markup from being interpreted as HTML.

In [14]:
import ipywidgets as widgets
from IPython.display import display, clear_output

chat_messages = new_conversation()
chat_log = widgets.Output(layout={"border": "1px solid #999", "height": "330px", "overflow_y": "auto"})
text_input = widgets.Text(placeholder="e.g., I want a flight to Paris", description="You:",
                          layout=widgets.Layout(width="75%"))
send_button = widgets.Button(description="Send", button_style="primary")
reset_button = widgets.Button(description="Reset chat")

def send_message(_=None):
    user_text = text_input.value.strip()
    if not user_text:
        return
    text_input.value = ""
    with chat_log:
        print("YOU:", user_text)
        try:
            answer = travel_agent(user_text, chat_messages)
            print("AGENT:", answer, "\n")
        except Exception as exc:
            print("ERROR:", str(exc), "\n")

def reset_chat(_=None):
    chat_messages.clear()
    chat_messages.extend(new_conversation())
    with chat_log:
        clear_output()
        print("Chat reset. Simulated booking records remain in BOOKING_LOG.")

send_button.on_click(send_message)
reset_button.on_click(reset_chat)
display(widgets.HTML("<h3>AI Travel Agent — Simulated Flight Booking</h3>"))
display(chat_log, widgets.HBox([text_input, send_button, reset_button]))

HTML(value='<h3>AI Travel Agent — Simulated Flight Booking</h3>')

Output(layout=Layout(border='1px solid #999', height='330px', overflow_y='auto'))

Part G — Implemented extension: hotel search and test evidence
Run the direct tests below, then use the interactive chat to test an actual LLM tool request. The direct tests verify the Python filtering logic without requiring Ollama. Chat-model execution must be run in Colab and shown to your instructor.

In [15]:
import json

# ASSESSMENT: Sample hotel catalog, NOT fixed customer details.
# The chat accepts any customer name and city; hotels are searched in this demo catalog.
# Simulated database of hotels
HOTELS_DB = [
    {"name": "Budget Inn London", "location": "London", "price": 80, "wifi": True},
    {"name": "Grand Hotel London", "location": "London", "price": 150, "wifi": True},
    {"name": "Cozy London Stay", "location": "London", "price": 90, "wifi": False},
    {"name": "Paris Luxury Suites", "location": "Paris", "price": 200, "wifi": True},
    {"name": "Paris Comfort Inn", "location": "Paris", "price": 110, "wifi": True}
]

def find_hotels(location: str, max_price: float, wifi_required: bool) -> dict:
    # Input validation
    if not isinstance(location, str) or not location.strip():
        return {"status": "error", "message": "Location must be a non-empty string."}

    if not isinstance(max_price, (int, float)) or max_price < 0:
        return {"status": "error", "message": "Price limit must be a positive number."}

    if not isinstance(wifi_required, bool):
        return {"status": "error", "message": "Wifi required must be a boolean value."}

    # Case-insensitive filtering
    location_lower = location.lower().strip()
    filtered = [
        h for h in HOTELS_DB
        if h["location"].lower() == location_lower
        and h["price"] <= max_price
        and (not wifi_required or h["wifi"] == wifi_required)
    ]

    return {
        "status": "success",
        "hotels": filtered,
        "count": len(filtered)
    }

# Register the new tool in the schemas list
if not any(t["function"]["name"] == "find_hotels" for t in tools):
    tools.append({
        "type": "function",
        "function": {
            "name": "find_hotels",
            "description": "Search for hotels in a specified location based on maximum price per night and Wi-Fi requirements.",
            "parameters": {
                "type": "object",
                "properties": {
                    "location": {"type": "string", "description": "Target city or location"},
                    "max_price": {"type": "number", "description": "Maximum budget per night"},
                    "wifi_required": {"type": "boolean", "description": "Whether Wi-Fi is mandatory"}
                },
                "required": ["location", "max_price", "wifi_required"],
                "additionalProperties": False
            }
        }
    })

# Register the new tool in the Python registry
TOOL_REGISTRY["find_hotels"] = find_hotels

# Deterministic validation: tests execute without the LLM
assert find_hotels("London", 100, True)["count"] == 1
assert find_hotels("london", 100, False)["count"] == 2
assert find_hotels("Paris", 120, True)["count"] == 1
assert find_hotels("London", 50, True)["count"] == 0
assert find_hotels("London", -1, True)["status"] == "error"
assert find_hotels("London", 100, "yes")["status"] == "error"
assert {t["function"]["name"] for t in tools} == {"book_flight", "find_hotels"}
assert "find_hotels" in TOOL_REGISTRY
print("PASS: 8 hotel filtering, validation, schema, and registry checks")
print("Example result:", json.dumps(find_hotels("London", 100, True), indent=2))

PASS: 8 hotel filtering, validation, schema, and registry checks
Example result: {
  "status": "success",
  "hotels": [
    {
      "name": "Budget Inn London",
      "location": "London",
      "price": 80,
      "wifi": true
    }
  ],
  "count": 1
}


In [16]:
import json

# --- Implement Option 2: find_activities ---
ACTIVITIES_DB = [
    {"name": "British Museum Tour", "location": "London", "category": "culture", "price": 15},
    {"name": "London Eye Flight", "location": "London", "category": "culture", "price": 40},
    {"name": "Thames Speedboat", "location": "London", "category": "adventure", "price": 50},
    {"name": "Louvre Museum Guide", "location": "Paris", "category": "culture", "price": 25},
    {"name": "Eiffel Tower Climb", "location": "Paris", "category": "adventure", "price": 35}
]

def find_activities(location: str, category: str, max_price: float) -> dict:
    if not isinstance(location, str) or not location.strip():
        return {"status": "error", "message": "Location must be a non-empty string."}
    if not isinstance(category, str) or not category.strip():
        return {"status": "error", "message": "Category must be a non-empty string."}
    if not isinstance(max_price, (int, float)) or max_price < 0:
        return {"status": "error", "message": "Price limit must be a positive number."}

    loc_lower = location.lower().strip()
    cat_lower = category.lower().strip()

    filtered = [
        act for act in ACTIVITIES_DB
        if act["location"].lower() == loc_lower
        and act["category"].lower() == cat_lower
        and act["price"] <= max_price
    ]
    return {
        "status": "success",
        "activities": filtered,
        "count": len(filtered)
    }

# --- Implement Option 3: Booking Cancellation support ---
# Let's adjust book_flight temporarily to add "booking_id"
_original_book_flight = book_flight
def book_flight(passenger_name: str, from_city: str, to_city: str, travel_date: str) -> dict:
    result = _original_book_flight(passenger_name, from_city, to_city, travel_date)
    if result.get("status") == "simulated_only":
        # FIX: Assign a unique reference so the customer can cancel later.
        from uuid import uuid4
        result["booking_id"] = f"DEMO-{uuid4().hex[:8].upper()}"
    return result

# Register updated function
TOOL_REGISTRY["book_flight"] = book_flight

def cancel_simulated_booking(booking_id: str) -> dict:
    if not isinstance(booking_id, str) or not booking_id.strip():
        return {"status": "error", "message": "Booking ID must be a valid string."}

    # FIX: Search both flight and hotel bookings in the same log.
    for booking in BOOKING_LOG:
        if booking.get("booking_id") == booking_id:
            if booking.get("status") == "simulated_cancelled":
                return {"status": "already_cancelled", "message": "This booking has already been cancelled."}
            booking["status"] = "simulated_cancelled"
            return {"status": "simulated_cancelled", "booking_id": booking_id, "message": "Simulated booking successfully cancelled."}

    return {"status": "not_found", "message": f"Booking ID {booking_id} not found."}

# --- Register tools in schema and registry ---
if not any(t["function"]["name"] == "find_activities" for t in tools):
    tools.append({
        "type": "function",
        "function": {
            "name": "find_activities",
            "description": "Search for local attractions or activities based on location, category, and maximum budget.",
            "parameters": {
                "type": "object",
                "properties": {
                    "location": {"type": "string"},
                    "category": {"type": "string"},
                    "max_price": {"type": "number"}
                },
                "required": ["location", "category", "max_price"],
                "additionalProperties": False
            }
        }
    })

if not any(t["function"]["name"] == "cancel_simulated_booking" for t in tools):
    tools.append({
        "type": "function",
        "function": {
            "name": "cancel_simulated_booking",
            "description": "Cancel an active simulated flight or hotel booking using its booking ID.",
            "parameters": {
                "type": "object",
                "properties": {
                    "booking_id": {"type": "string"}
                },
                "required": ["booking_id"],
                "additionalProperties": False
            }
        }
    })

TOOL_REGISTRY["find_activities"] = find_activities
TOOL_REGISTRY["cancel_simulated_booking"] = cancel_simulated_booking

# --- Run Tests ---
# Option 2: activity filtering, category and price validation
assert find_activities("London", "culture", 30)["count"] == 1
assert find_activities("london", "CULTURE", 40)["count"] == 2
assert find_activities("Paris", "adventure", 35)["count"] == 1
assert find_activities("London", "adventure", 20)["count"] == 0
assert find_activities("", "culture", 30)["status"] == "error"
assert find_activities("London", "culture", -1)["status"] == "error"
print("PASS: 6 activity tests")
print("Activity result:", json.dumps(find_activities("London", "culture", 40), indent=2))

# Option 3: create, cancel, repeat cancellation, invalid ID
BOOKING_LOG.clear()  # clean isolated test log
booking = book_flight("Demo Student", "London", "Paris", "2030-06-15")
assert booking["booking_id"].startswith("DEMO-")
assert booking["status"] == "simulated_only"
assert cancel_simulated_booking(booking["booking_id"])["status"] == "simulated_cancelled"
assert BOOKING_LOG[0]["status"] == "simulated_cancelled"
assert cancel_simulated_booking(booking["booking_id"])["status"] == "already_cancelled"
assert cancel_simulated_booking("DEMO-999")["status"] == "not_found"
assert cancel_simulated_booking("")["status"] == "error"
assert {t["function"]["name"] for t in tools} == {
    "book_flight", "find_hotels", "find_activities", "cancel_simulated_booking"}
assert all(name in TOOL_REGISTRY for name in ("find_activities", "cancel_simulated_booking"))
print("PASS: 9 booking cancellation / tool registration checks")
print("Cancellation example:", json.dumps(BOOKING_LOG[0], indent=2))
BOOKING_LOG.clear()  # avoid interfering with live chat demonstration

PASS: 6 activity tests
Activity result: {
  "status": "success",
  "activities": [
    {
      "name": "British Museum Tour",
      "location": "London",
      "category": "culture",
      "price": 15
    },
    {
      "name": "London Eye Flight",
      "location": "London",
      "category": "culture",
      "price": 40
    }
  ],
  "count": 2
}
PASS: 9 booking cancellation / tool registration checks
Cancellation example: {
  "status": "simulated_cancelled",
  "passenger_name": "Demo Student",
  "from_city": "London",
  "to_city": "Paris",
  "travel_date": "2030-06-15",
  "message": "SIMULATION ONLY: no flight has actually been booked.",
  "booking_id": "DEMO-1074E368"
}


In [17]:
# ASSESSMENT EXTENSION: Simulated hotel booking for ANY customer (no prefilled name or city).
# Hotel searching above remains unchanged. Booking creates a reference in BOOKING_LOG.
from datetime import date
from uuid import uuid4

def book_hotel(passenger_name: str, hotel_name: str, location: str,
               check_in: str, check_out: str) -> dict:
    # FIX: Validate hotel-specific fields, not flight departure/destination fields.
    fields = (passenger_name, hotel_name, location, check_in, check_out)
    if not all(isinstance(v, str) and v.strip() for v in fields):
        return {"status": "error", "message": "Please provide customer name, hotel, city, check-in and check-out dates."}
    try:
        start, end = date.fromisoformat(check_in), date.fromisoformat(check_out)
    except ValueError:
        return {"status": "error", "message": "Dates must use YYYY-MM-DD."}
    if start < date.today() or end <= start:
        return {"status": "error", "message": "Check-in must not be in the past and check-out must be after check-in."}
    match = next((h for h in HOTELS_DB if h["name"].casefold() == hotel_name.strip().casefold()
                  and h["location"].casefold() == location.strip().casefold()), None)
    if match is None:
        return {"status": "error", "message": "Hotel not found in the simulated catalog; use find_hotels first."}
    record = {"status": "simulated_only", "booking_id": "HTL-" + uuid4().hex[:8].upper(),
              "booking_type": "hotel", "passenger_name": passenger_name.strip(),
              "hotel_name": match["name"], "location": match["location"],
              "check_in": check_in, "check_out": check_out,
              "total_price": (end - start).days * match["price"],
              "message": "SIMULATION ONLY: no hotel has actually been booked."}
    BOOKING_LOG.append(record)
    return record

# Register the hotel-booking tool with the LLM and Python dispatcher.
if not any(t["function"]["name"] == "book_hotel" for t in tools):
    tools.append({"type": "function", "function": {
        "name": "book_hotel", "description": "Create a simulated hotel booking for a customer after they choose a hotel from find_hotels.",
        "parameters": {"type": "object", "properties": {
            "passenger_name": {"type": "string"}, "hotel_name": {"type": "string"},
            "location": {"type": "string"}, "check_in": {"type": "string", "description": "YYYY-MM-DD"},
            "check_out": {"type": "string", "description": "YYYY-MM-DD"}},
            "required": ["passenger_name", "hotel_name", "location", "check_in", "check_out"],
            "additionalProperties": False}}})
TOOL_REGISTRY["book_hotel"] = book_hotel

# ASSESSMENT TESTS: Names and cities below are test data only, never chat defaults.
assert find_hotels("Paris", 110, True)["count"] == 1
assert find_hotels("London", 50, True)["count"] == 0
assert book_hotel("Sample Customer", "Paris Comfort Inn", "Paris", "2030-06-15", "2030-06-17")["status"] == "simulated_only"
hotel_id = BOOKING_LOG[-1]["booking_id"]
assert cancel_simulated_booking(hotel_id)["status"] == "simulated_cancelled"
assert cancel_simulated_booking(hotel_id)["status"] == "already_cancelled"
assert cancel_simulated_booking("UNKNOWN-REFERENCE")["status"] == "not_found"
assert "book_hotel" in TOOL_REGISTRY
BOOKING_LOG.clear()  # Test bookings are cleared; customer input is never hardcoded.
print("PASS: hotel search, hotel booking, cancellation and tool registration tests")


PASS: hotel search, hotel booking, cancellation and tool registration tests


## Part G — Exercise (5 Marks): Extending the Travel Agent with a New Tool

**Task: **Extend the existing Travel Agent by implementing ONE additional tool of your choice:

**Option 1** — Find Hotels: Search for hotels in a specified location based on the user's budget and Wi-Fi preference.

**Option 2** — Find Activities: Search for activities in a specified location based on the user's preferred category and maximum budget.

**Option 3 **— Cancel Booking: Cancel an existing simulated flight booking using its booking reference.

Your Travel Agent should automatically identify and execute the appropriate tool based on the user's request.

##Tips

### 1 Define a new Python function: Follow the existing book_flight() example. Depending on your chosen tool, use the following parameters:

find_hotels(location: str, max_price: float, wifi_required: bool) — filter hotels by location, maximum nightly price and Wi-Fi availability.

find_activities(location: str, category: str, max_price: float) — filter activities by location, category and maximum activity price.

cancel_simulated_booking(booking_id: str) — find the booking reference in BOOKING_LOG, update its status to simulated_cancelled and handle invalid or previously cancelled booking IDs.

### 2 Update the tools JSON schema: Add the new function's name, description, parameters, data types and required fields so the LLM understands when to use it. For cancellation, the required parameter is booking_id (e.g., DEMO-001).

### 3 Modify travel_agent(): Extend the existing tool-calling logic to recognise the new function, extract the required arguments and execute the corresponding Python function. If your tutorial uses TOOL_REGISTRY, register the new function there as well.

### 4 Update the system prompt: Inform the agent about its additional capability and instruct it to ask follow-up questions when required information is missing.

### Test your implementation: Use the existing chat interface to verify that the agent selects and executes the correct tool. For cancellation, first create a simulated booking, then test successful cancellation, repeated cancellation and an invalid booking reference.

## Assessment Criteria (1 *5)

1: Correct implementation of the new Python function

2: Correct JSON tool definition and registration

3: Successful integration into travel_agent()

4: Correct handling of parameters, filtering or booking status

5: Successful testing through the chat interface

**Submission:** Demostarte the implemenation to your lab instructor and Submit the updated notbook in PDF/HTML containing the additional tool and evidence of successful execution.

### Troubleshooting
- **Ollama connection refused:** rerun the server-start cell and confirm `/api/tags` is reachable.
- **Model not found:** rerun `ollama pull qwen2.5:3b`.
- **Out of memory / very slow:** use a GPU runtime if available; alternatively try `qwen2.5:1.5b` with the corresponding pull command, noting reduced tool-call reliability.
- **Tool calls missing:** check the model, `tools` schema and `tool_choice="auto"`; small models do not guarantee identical tool-selection behaviour.
- **OpenAI 429 quota error:** the optional paid API configuration needs API billing credits independent of a ChatGPT subscription.

**Further reading:** [Ollama OpenAI compatibility](https://docs.ollama.com/api/openai-compatibility) · [OpenAI function calling](https://platform.openai.com/docs/guides/function-calling) · [Google Colab](https://colab.research.google.com/).

### Assessment completion notes (5 marks)

1. **Python functions:** `find_hotels`, `find_activities`, `cancel_simulated_booking`, and `book_hotel` validate inputs and return structured results.
2. **JSON schemas:** Each tool has its own name, parameters, types and required fields; all tools are registered in `TOOL_REGISTRY`.
3. **Agent integration:** The **FIX** in Part D reads required fields from the *selected tool schema* instead of checking flight parameters for every tool.
4. **Correct behavior:** Hotel filtering respects city, budget and Wi-Fi; flight and hotel bookings get cancellation references; repeat cancellation and unknown IDs are handled.
5. **Testing:** The assertion cells provide deterministic evidence. **Run the chat interface in Colab after running Part G**, then demonstrate flight booking, hotel search/booking and cancellation to your instructor. Live Ollama chat execution is not verified here.

**Customer data:** No customer name, origin or destination is fixed in the agent. Example names and locations in test cells are sample fixtures only. Bookings are simulated, not real purchases.
